# Volatility modeling notebook (refactored)

This version keeps your notebook workflow, but makes it more structured:
- configuration up front
- reusable data-loading helpers
- a cleaner pipeline class
- explicit execution steps
- optional tuning separated from the default run

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use("fivethirtyeight")

In [ ]:
@dataclass
class Config:
    data_path: str = "./data/df_features.csv"
    start_date: str = "2019-04-30"
    end_date: str = "2026-03-26"
    target_col: str = "targetvol21d"
    date_col: str = "tradingdate"
    test_size: float = 0.15
    n_splits: int = 5
    random_state: int = 42
    run_optuna: bool = False
    optuna_trials: int = 30

CONFIG = Config()

In [ ]:
def load_modeling_data(config: Config) -> pd.DataFrame:
    df = pd.read_csv(config.data_path)
    df[config.date_col] = pd.to_datetime(df[config.date_col])
    df = df.sort_values(config.date_col).reset_index(drop=True)
    df = df[
        (df[config.date_col] >= config.start_date)
        & (df[config.date_col] <= config.end_date)
    ].reset_index(drop=True).copy()
    print(f"Loaded {len(df):,} rows from {df[config.date_col].min().date()} to {df[config.date_col].max().date()}")
    return df


def validate_required_columns(df: pd.DataFrame, required_cols: List[str]) -> None:
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

In [ ]:
class VolatilityRiskPipeline:
    """Structured pipeline for 21-day forward volatility modeling."""

    def __init__(
        self,
        df: pd.DataFrame,
        target_col: str = "targetvol21d",
        date_col: str = "tradingdate",
        test_size: float = 0.15,
        random_state: int = 42,
    ):
        self.df = df.copy()
        self.df[date_col] = pd.to_datetime(self.df[date_col])
        self.df = self.df.sort_values(date_col).reset_index(drop=True)

        self.target = target_col
        self.date_col = date_col
        self.random_state = random_state
        self.model = None
        self.explainer = None
        self.best_params_ = None
        self.cv_results_ = None
        self.test_results_ = None

        split_idx = int(len(self.df) * (1 - test_size))
        self.trainval_df = self.df.iloc[:split_idx].reset_index(drop=True)
        self.test_df = self.df.iloc[split_idx:].reset_index(drop=True)

        self.feature_groups = {
            "PriceDynamics": ["volrolling21d", "msftreturn", "distfromma200", "volgarmanklass", "amihudratio", "msftvsmarket", "msftvstech"],
            "MacroRegime": ["vixlevel", "vix5dtrend", "yield10ylevel", "yield10ydelta5d", "qqqvol21d", "volsurge"],
            "NLPSentiment": ["preppeddispersion", "preppedneutral", "qasentiment", "qadispersion", "qaneutral", "RiskCombinedMean", "MDANeutrality", "RiskCombinedStd", "RiskCombinedNeutrality", "MDADelta", "MDASentiment", "RiskDelta", "preppedsentiment"],
            "FundamentalAccounting": ["roa", "debttoasset", "fcfmargin", "netincomeqoq", "cashcoverage"],
            "Temporal": ["dayssinceearnings", "dayssincefiling"],
        }

        self.all_features = [feat for group in self.feature_groups.values() for feat in group if feat in self.df.columns]

        print("Data split complete")
        print(f"Train/validation rows: {len(self.trainval_df):,} | ends {self.trainval_df[self.date_col].max().date()}")
        print(f"Holdout test rows:      {len(self.test_df):,} | starts {self.test_df[self.date_col].min().date()}")

    def set_active_features(self, custom_feature_list: List[str]) -> None:
        missing = [f for f in custom_feature_list if f not in self.df.columns]
        if missing:
            raise ValueError(f"These selected features are missing from dataframe: {missing}")
        old_count = len(self.all_features)
        self.all_features = custom_feature_list
        print(f"Feature space pruned: {old_count} -> {len(self.all_features)}")

    @staticmethod
    def _rmse(y_true, y_pred) -> float:
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))

    def _default_lgbm_params(self) -> Dict:
        return {
            "objective": "regression",
            "metric": "rmse",
            "learning_rate": 0.01,
            "n_estimators": 2000,
            "max_depth": 5,
            "num_threads": 6,
            "colsample_bytree": 0.6,
            "reg_alpha": 1.0,
            "reg_lambda": 1.0,
            "verbosity": -1,
            "seed": self.random_state,
        }

    def optimize_lgbm_hyperparameters(self, n_trials: int = 30, n_splits: int = 5) -> Dict:
        X = self.trainval_df[self.all_features]
        y = self.trainval_df[self.target]
        tscv = TimeSeriesSplit(n_splits=n_splits)

        def objective(trial):
            params = {
                "objective": "regression",
                "metric": "rmse",
                "learning_rate": 0.02,
                "n_estimators": 2000,
                "verbosity": -1,
                "seed": self.random_state,
                "max_depth": trial.suggest_int("max_depth", 3, 6),
                "num_leaves": trial.suggest_int("num_leaves", 7, 31),
                "min_child_samples": trial.suggest_int("min_child_samples", 20, 60),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
                "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
            }
            scores = []
            for train_idx, val_idx in tscv.split(X):
                X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
                X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
                model = lgb.LGBMRegressor(**params)
                model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(25, verbose=False)])
                preds = model.predict(X_val)
                scores.append(self._rmse(y_val, preds))
            return float(np.mean(scores))

        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=n_trials)
        self.best_params_ = study.best_params
        print(f"Best CV RMSE: {study.best_value:.4f}")
        return study.best_params

    def compare_model_progression(self, n_splits: int = 5, tuned_params: Optional[Dict] = None) -> pd.DataFrame:
        tscv = TimeSeriesSplit(n_splits=n_splits)
        X = self.trainval_df[self.all_features]
        y = self.trainval_df[self.target]
        price_features = [f for f in self.feature_groups["PriceDynamics"] if f in self.all_features]

        params = self._default_lgbm_params()
        if tuned_params:
            params.update(tuned_params)

        rows, naive_scores, price_scores, alpha_scores = [], [], [], []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X), start=1):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

            naive_preds = X_val["volrolling21d"]
            naive_rmse = self._rmse(y_val, naive_preds)
            naive_scores.append(naive_rmse)

            model_price = lgb.LGBMRegressor(**params)
            model_price.fit(X_train[price_features], y_train, eval_set=[(X_val[price_features], y_val)], callbacks=[lgb.early_stopping(15, verbose=False)])
            price_preds = model_price.predict(X_val[price_features])
            price_rmse = self._rmse(y_val, price_preds)
            price_scores.append(price_rmse)

            model_alpha = lgb.LGBMRegressor(**params)
            model_alpha.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(15, verbose=False)])
            alpha_preds = model_alpha.predict(X_val)
            alpha_rmse = self._rmse(y_val, alpha_preds)
            alpha_scores.append(alpha_rmse)

            rows.append({"fold": fold, "naive_rmse": naive_rmse, "price_rmse": price_rmse, "alpha_rmse": alpha_rmse})
            print(f"Fold {fold}: Naive={naive_rmse:.4f} | Price={price_rmse:.4f} | Alpha={alpha_rmse:.4f}")

        self.cv_results_ = pd.DataFrame(rows)
        print(f"Average Naive RMSE: {np.mean(naive_scores):.4f}")
        print(f"Average Price RMSE: {np.mean(price_scores):.4f}")
        print(f"Average Alpha RMSE: {np.mean(alpha_scores):.4f}")

        self.model = lgb.LGBMRegressor(**params)
        self.model.fit(X, y)
        return self.cv_results_

    def generate_shap_explanations(self, top_n: int = 15) -> None:
        if self.model is None:
            raise ValueError("Model must be trained first.")
        X = self.trainval_df[self.all_features]
        lgb.plot_importance(self.model, max_num_features=top_n, importance_type="gain", figsize=(10, 6), title="LightGBM feature importance")
        plt.tight_layout()
        plt.show()
        self.explainer = shap.TreeExplainer(self.model)
        shap_values = self.explainer.shap_values(X)
        shap.summary_plot(shap_values, X)

    def generate_local_shap_force(self, index: int = 0):
        if self.explainer is None:
            raise ValueError("Run generate_shap_explanations() first.")
        X_test = self.test_df[self.all_features]
        row_2d = X_test.iloc[[index]]
        row_1d = X_test.iloc[index]
        shap.initjs()
        shap_values = self.explainer.shap_values(row_2d)
        return shap.force_plot(self.explainer.expected_value, shap_values[0], row_1d)

    
def evaluate_on_test_set(self) -> pd.DataFrame:
    if self.model is None:
        raise ValueError("Train the model before test evaluation.")

    X_test = self.test_df[self.all_features]
    y_test = self.test_df[self.target]
    preds = self.model.predict(X_test)

    test_rmse = self._rmse(y_test, preds)
    test_mae = float(mean_absolute_error(y_test, preds))
    mean_error = float(np.mean(preds - y_test))

    print(f"Test RMSE: {test_rmse:.4f}")
    print(f"Test MAE: {test_mae:.4f}")
    print(f"Mean Error: {mean_error:.4f}")

    plt.figure(figsize=(12, 6))
    plt.plot(self.test_df[self.date_col], y_test, label="Actual 21d vol", color="#d3d3d3", linewidth=2)
    plt.plot(self.test_df[self.date_col], preds, label="Predicted", color="#2e5597", linewidth=2)
    plt.title("Out-of-sample tracking", fontweight="bold")
    plt.xlabel("Trading date")
    plt.ylabel("Annualized volatility")
    plt.legend()
    plt.tight_layout()
    plt.show()

    errors = preds - y_test
    plt.figure(figsize=(8, 5))
    plt.hist(errors, bins=30, color="#2e5597", alpha=0.7, edgecolor="white")
    plt.axvline(0, color="black", linestyle="-", linewidth=2, label="Zero Error")
    plt.axvline(mean_error, color="#d9534f", linestyle="--", linewidth=2, label=f"Mean Bias {mean_error:.4f}")
    plt.title("Residual distribution", fontweight="bold")
    plt.xlabel("Prediction error")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.show()

    threshold_90 = np.percentile(y_test, 90)
    tail_mask = y_test >= threshold_90
    tail_rmse = self._rmse(y_test[tail_mask], preds[tail_mask])

    current_vol = self.test_df["volrolling21d"]
    actual_direction = np.sign(y_test - current_vol)
    predicted_direction = np.sign(preds - current_vol)
    hit_rate = float(np.mean(actual_direction == predicted_direction) * 100)

    print(f"Tail RMSE (p90+): {tail_rmse:.4f}")
    print(f"Directional Hit Rate: {hit_rate:.2f}%")

    calib_df = pd.DataFrame({"Actual": y_test, "Predicted": preds})
    calib_df["Quintile"] = pd.qcut(
        calib_df["Predicted"].rank(method="first"),
        q=5,
        labels=False,
    ) + 1
    quintile_stats = calib_df.groupby("Quintile").mean(numeric_only=True)

    plt.figure(figsize=(8, 5))
    x = np.arange(5)
    width = 0.35
    plt.bar(x - width / 2, quintile_stats["Predicted"], width, label="Predicted", color="#2e5597")
    plt.bar(x + width / 2, quintile_stats["Actual"], width, label="Actual", color="#d3d3d3")
    plt.xticks(x, ["Q1 Safe", "Q2", "Q3", "Q4", "Q5 Danger"])
    plt.xlabel("Predicted risk quintile")
    plt.ylabel("Annualized volatility")
    plt.title("Holdout calibration")
    plt.legend()
    plt.tight_layout()
    plt.show()

    calibration_table = quintile_stats.rename(columns={
        "Predicted": "avg_predicted_vol",
        "Actual": "avg_actual_vol",
    }).reset_index()
    print("Calibration table:")
    display(calibration_table)

    self.test_results_ = pd.DataFrame([{
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "mean_error": mean_error,
        "tail_rmse_p90": tail_rmse,
        "directional_hit_rate": hit_rate,
    }])
    return self.test_results_


In [ ]:
PRUNED_FEATURES = [
    "volrolling21d",
    "msftreturn",
    "distfromma200",
    "volgarmanklass",
    "amihudratio",
    "msftvsmarket",
    "msftvstech",
    "qadispersion",
    "vixlevel",
    "fcfmargin",
    "yield10ylevel",
    "preppedneutral",
    "dayssinceearnings",
    "cashcoverage",
    "qqqvol21d",
    "qasentiment",
    "preppedsentiment",
]

In [ ]:
df = load_modeling_data(CONFIG)
validate_required_columns(df, [CONFIG.date_col, CONFIG.target_col] + PRUNED_FEATURES)

pipeline = VolatilityRiskPipeline(
    df=df,
    target_col=CONFIG.target_col,
    date_col=CONFIG.date_col,
    test_size=CONFIG.test_size,
    random_state=CONFIG.random_state,
)

pipeline.set_active_features(PRUNED_FEATURES)

In [ ]:
cv_results = pipeline.compare_model_progression(n_splits=CONFIG.n_splits)
cv_results

In [ ]:
if CONFIG.run_optuna:
    best_params = pipeline.optimize_lgbm_hyperparameters(
        n_trials=CONFIG.optuna_trials,
        n_splits=CONFIG.n_splits,
    )
    cv_results_tuned = pipeline.compare_model_progression(
        n_splits=CONFIG.n_splits,
        tuned_params=best_params,
    )
    cv_results_tuned

In [ ]:
pipeline.generate_shap_explanations()

In [ ]:
# Adjust the index if you want to inspect a different day
pipeline.generate_local_shap_force(index=100)

In [ ]:
test_results = pipeline.evaluate_on_test_set()
test_results